# 20 — V6.5 external expansion staging → `v6p5_ext_<hash>`
Sources (settings unchanged from the last Kaggle notebook):
* **Open Images V7 TRAIN** Balloon / Kite / Plastic bag / Bird / Person — appearance only,
  `detections_only`, local GrabCut mask, 250/class, seed 20261007;
* **MPCD** broken/fracture → `broken_strand`;
* **Power Equipment** `dx_dg→broken_strand`, `yw→foreign_object`, `nw→bird_nest`, `dx_sg` review-only;
* **synthetic** relation-aware foreign objects on internal TRAIN backgrounds, 100/subtype, TRAIN only.
Then exact-SHA + pHash QA, review manifest, contact sheets, deterministic packaging, freeze, Camber upload.
Each stage checkpoints (per class / per subtype) and resumes after a restart.

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
outcomes = {}
for s in ["parent", "openimages", "mpcd", "power_equipment", "synthetic", "dedup"]:
    outcomes[s] = pipeline.run_stage(s, force=FORCE_REBUILD, dry_run=DRY_RUN)
    print(outcomes[s])

In [ ]:
d = store.load("dedup").summary
print("exact SHA duplicates vs V6.2E (HARD gate):", d["exact_duplicates"])
print("pHash candidates vs VAL/TEST (review):   ", d["phash_candidates"])

In [ ]:
outcome = pipeline.run_stage("external_freeze", force=FORCE_REBUILD, dry_run=DRY_RUN)
print(outcome)
display(pd.Series(outcome.summary, dtype=object).to_frame("value"))

In [ ]:
contacts = sorted((paths.qa / "v6p5_external" / "contacts").glob("*.jpg"))
print(len(contacts), "contact sheets")
for p in contacts[:12]:
    print(p.name); display(IPImage(filename=str(p), width=640))